# Final Milestone — LLM Comparison and Tool Demo

For the final milestone, we have decided to do the following:
1. Load retrievers and RAG pipeline components
2. Compare two LLMs: `Meta-Llama-3-8B-Instruct` (HF API) vs `Qwen3 1.7B` (local via Ollama)
3. Run 5 queries through both models with identical retriever and prompt
4. Demonstrate Tavily web search tool on 3 example queries

**Note:** We initially planned to run both models via the HF Inference API, but `Qwen/Qwen3.5-2B` is not hosted by any HF provider. We use Ollama to run Qwen3 locally instead, which also gives us an interesting API-vs-local comparison axis.

In [5]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")

True

In [6]:
from src.bm25 import BM25Retriever
from src.semantic import SemanticRetriever

INDICES = Path.cwd().parent / "indices"

bm25 = BM25Retriever()
bm25.load(str(INDICES / "bm25_index.pkl"))

semantic = SemanticRetriever()
semantic.load(str(INDICES / "faiss_index"))

print(f"BM25 corpus: {len(bm25.corpus):,} products")
print(f"Semantic corpus: {len(semantic.corpus):,} products")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5855.58it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BM25 corpus: 112,590 products
Semantic corpus: 112,590 products


## Step 1.2: LLM Comparison

We compare two models using **identical** retrieval (Hybrid/RRF) and prompt (`strict_citation`):

| Model | Parameters | Family | Provider |
|---|---|---|---|
| `meta-llama/Meta-Llama-3-8B-Instruct` | 8B | Meta Llama 3 | HF Inference API (remote) |
| `qwen3:1.7b` | 1.7B | Alibaba Qwen 3 | Ollama (local) |

The ~5x parameter difference and different model families should reveal quality-vs-size tradeoffs in citation accuracy, completeness, and fluency.

In [7]:
from src.rag_pipeline import RAGPipeline, load_llm
from langchain_ollama import ChatOllama

MODEL_A = "meta-llama/Meta-Llama-3-8B-Instruct"
MODEL_B = "qwen3:1.7b"

llm_a = load_llm(model_id=MODEL_A)
llm_b = ChatOllama(model=MODEL_B)

print(f"Model A: {MODEL_A} (HF Inference API)")
print(f"Model B: {MODEL_B} (Ollama, local)")

Model A: meta-llama/Meta-Llama-3-8B-Instruct (HF Inference API)
Model B: qwen3:1.7b (Ollama, local)


In [8]:
pipeline_a = RAGPipeline(
    bm25=bm25, semantic=semantic,
    retriever_name="Hybrid", prompt_name="strict_citation",
    llm=llm_a, top_k=5,
)
pipeline_b = RAGPipeline(
    bm25=bm25, semantic=semantic,
    retriever_name="Hybrid", prompt_name="strict_citation",
    llm=llm_b, top_k=5,
)

comparison_queries = [
    "vitamin C serum for brightening",
    "what helps with mild acne and post-acne marks?",
    "gentle face wash that won't irritate sensitive skin",
    "best skincare routine for oily acne-prone teenage skin under $30",
    "what helps with sun damage on fair skin around the eyes",
]

results_comparison = []
for q in comparison_queries:
    print("=" * 80)
    print("QUERY:", q)
    print("=" * 80)

    result_a = pipeline_a.answer(q)
    print("--- Model A (Llama-3-8B, HF API) ---")
    print(result_a["answer"])
    print()

    result_b = pipeline_b.answer(q)
    print("--- Model B (Qwen3 1.7B, Ollama) ---")
    print(result_b["answer"])
    print()

    results_comparison.append({
        "query": q,
        "answer_a": result_a["answer"],
        "answer_b": result_b["answer"],
        "sources_a": result_a["sources"],
        "sources_b": result_b["sources"],
    })

QUERY: vitamin C serum for brightening
--- Model A (Llama-3-8B, HF API) ---
Based on the provided reviews and metadata, I found several vitamin C serums for brightening:

1. Ultra-Brightening Vitamin C Serum, Advanced Formula with Vitamin C Concentrate, Vitamin E, Green Tea, Aloe Vera Extracts to Revive Dull Sun-Damaged Skin, Nourishing Skincare for Healthy Glowing Complexion 2 fl. oz. [B09T4FW4S9]
2. Eva St. Claire Vitamin C Serum. Anti-aging brightening serum with Vitamin C, Ferulic Acid, Collagen, Aloe Vera for dark spots, skin discoloration. 1 fl oz (30ml) [B07PC2CNK7]
3. Essano Vitamin C Brightening Serum - Boost and Brighten, 30ml [B0874HTLRB]
4. Mererke_Pretty Vitamin C Serum for Face: Vitamin C 20 Facial and Under Eye Serum with Hyaluronic Acid - Wrinkle Remover Serum to Even and Tone Skin - Anti Aging and Brightening Skin Care Serums - 1 F [B01N6ADBNM]
5. StBotanica Facial Day Youth Serum - 30ml - with Natural SPF & Vitamin C for Skin Brightening, Fairness [B07BK5TC2X]

These 

In [9]:
import json

COMPARISON_OUT = Path.cwd().parent / "data" / "eval_outputs" / "llm_comparison.json"
COMPARISON_OUT.parent.mkdir(parents=True, exist_ok=True)
COMPARISON_OUT.write_text(json.dumps(results_comparison, indent=2))
print(f"Saved {len(results_comparison)} comparison results to {COMPARISON_OUT}")

Saved 5 comparison results to /Users/williamchong/Documents/UBC_MDS/Block6/575advml/DSCI_575_project_willchh_jiromig/data/eval_outputs/llm_comparison.json


## Step 2: Tool Integration Demo (Tavily Web Search)

We demonstrate the Tavily web search tool (`src/tools.py`) on 3 queries where the corpus context alone may be insufficient — e.g., queries about current prices, recent product launches, or ingredient safety information that reviews don't cover.

In [10]:
from src.tools import web_search

tool_queries = [
    "Is retinol safe to use with vitamin C serum?",
    "best drugstore sunscreen 2025 dermatologist recommended",
    "CeraVe moisturizer recall or safety issues",
]

tool_results = []
for q in tool_queries:
    print("QUERY:", q)
    print("-" * 60)

    # RAG answer (from corpus only)
    rag_result = pipeline_a.answer(q)
    print("RAG-only answer (first 300 chars):")
    print(rag_result["answer"][:300])
    print()

    # Web search result
    web_result = web_search.invoke({"query": q})
    print("Tavily web search result (first 500 chars):")
    print(web_result[:500] if web_result else "(no results)")
    print()

    tool_results.append({
        "query": q,
        "rag_answer": rag_result["answer"],
        "web_result": web_result,
    })

TOOL_OUT = Path.cwd().parent / "data" / "eval_outputs" / "tool_demo.json"
TOOL_OUT.write_text(json.dumps(tool_results, indent=2))
print(f"Saved {len(tool_results)} tool demo results to {TOOL_OUT}")

QUERY: Is retinol safe to use with vitamin C serum?
------------------------------------------------------------
RAG-only answer (first 300 chars):
I don't have enough information.

Tavily web search result (first 500 chars):
Vitamin C and retinol are two of the most high-profile ingredients in the world of skincare, thanks to the powerful skin benefits both ingredients provide. Anti-ageing properties are some of the most sought-after effects from skincare users and both retinol and vitamin C are known to help maintain the appearance of youthful skin. Given that both these ingredients can help lead to brighter, more even and younger-looking skin, it’s no surprise that questions arise about using retinol and vitamin C

QUERY: best drugstore sunscreen 2025 dermatologist recommended
------------------------------------------------------------
RAG-only answer (first 300 chars):
Based on the provided reviews, I don't have enough information to determine the best dermatologist-recommended dr